# Ders 11 - Ajanlar Arası (A2A) Protokolü


## Kurulum


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## What is the A2A Protocol?

The **Agent-to-Agent (A2A) protocol** is an open standard that enables AI agents to communicate,
discover each other, and collaborate — even when they are built on different frameworks or hosted
by different services.

Key concepts:

- **Discovery** – Agents publish an *Agent Card* that describes their capabilities, making it
  easy for other agents (or orchestrators) to find the right specialist for a task.
- **Message Passing** – Agents exchange structured messages through a common protocol, so a
  request from one agent can be understood and fulfilled by another regardless of its internal
  implementation.
- **Task Lifecycle** – A2A defines states such as *submitted*, *working*, *completed*, and
  *failed*, giving the orchestrator full visibility into how a delegated task is progressing.

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents
into a workflow where each agent contributes its expertise and passes results to the next.


## Uzmanlaşmış Seyahat Acenteleri Oluşturma


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## İş Akışı Yoluyla Çoklu Ajan İşbirliği

Üç ajanı, A2A mesaj iletimini yansıtan ardışık bir iş akışına bağlıyoruz:

1. **CurrencyExchangeAgent** kullanıcı isteğini alır ve döviz rehberliği sağlar.
2. **ActivityPlannerAgent** zenginleştirilmiş bağlamı alır ve etkinlik önerileri ekler.
3. **TravelManagerAgent** her iki girdiyi nihai bir seyahat özeti haline getirir.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Üretimde A2A'yı Anlamak

Üretim ortamında A2A protokolü güçlü çapraz servis senaryolarının kilidini açar:

| Yetkinlik | Açıklama |
|---|---|
| **Çerçeveler arası birlikte çalışabilirlik** | Bir çerçeve ile oluşturulmuş bir ajan, başka herhangi bir A2A uyumlu çerçeve ile oluşturulmuş ajana görevler devredebilir, gerçek çapraz organizasyon birlikte çalışabilirliği sağlar. |
| **Servis sınırları** | Ajanlar ayrı mikroservislerde, bulut bölgelerinde veya hatta farklı organizasyonlarda bulunabilirken hala sorunsuz çalışabilir. |
| **Dinamik keşif** | Bir düzenleyici çalışma zamanında bir Ajan Kartı kaydını sorgulayarak belirli bir alt görev için en uygun uzmana ulaşabilir. |
| **Akış ve bildirim göndermeleri** | A2A, gerçek zamanlı ilerleme güncellemeleri için Sunucu Gönderilen Olayları (SSE) ve uzun süren görevler için bildirim göndermeleri destekler. |

Yukarıda oluşturduğumuz iş akışı, bu modelin basitleştirilmiş, süreç içi bir versiyonudur. Gerçek bir
dağıtımda her ajan bir HTTP uç noktası sağlar, bir Ajan Kartı yayınlar ve
A2A JSON-RPC protokolü ile iletişim kurar.


## Summary

In this lesson you learned:

1. **What the A2A protocol is** — an open standard for agent-to-agent discovery, messaging,
   and task management.
2. **How to create specialized agents** — a Currency Exchange agent, an Activity Planner agent,
   and a Travel Manager orchestrator.
3. **How to wire agents into a workflow** — using `WorkflowBuilder` to model sequential
   message passing between agents.
4. **How A2A works in production** — enabling cross-framework, cross-service collaboration
   with dynamic discovery and streaming updates.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba sarf etsek de, otomatik çevirilerin hata veya yanlışlık içerebileceğini lütfen unutmayınız. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek yanlış anlamalardan veya yanlış yorumlamalardan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
